# 🎵 Music Generation with an LSTM (TensorFlow / Keras)

**Assignment notebook – RNNs, LSTMs and Sequence Modelling**

In this notebook you will build an end-to-end system that *learns from real folk tunes* and then *composes new melodies* on its own.

## Pipeline at a glance

```
MIDI files
    ↓
Extract notes
    ↓
Convert notes to integers
    ↓
Create sequences  (input: 50 events → target: the 51st event)
    ↓
Embedding
    ↓
LSTM
    ↓
Dense + Softmax
    ↓
Predict next note
    ↓
Generate sequence (autoregressively)
    ↓
MIDI file  (🎧 play + download)
```

## 1. Introduction

**What does "music generation" mean?**
We train a model on existing music, so that it learns the *patterns* of that music (which notes tend to follow which). Then we ask the model to continue a melody, one note at a time, producing music that **did not exist before** but sounds *in the style* of the training data.

**Why can music be treated as a sequence?**
Music unfolds in time. Whether a note sounds "right" depends on the notes that came *before* it – just like the next word in a sentence depends on the previous words. This is exactly the setting where RNNs/LSTMs are used.

**What will the LSTM predict?**
Given the last 50 musical events, the LSTM predicts a *probability for every possible next event*. We then pick (sample) one event, append it to the melody, and repeat. This is the same idea as next-word prediction in language models.

> **Language model analogy:** words → notes, sentences → melodies, vocabulary → set of distinct notes.

## 2. Install dependencies

Colab already has TensorFlow, NumPy and Matplotlib. We only need to add:

* `pretty_midi` – read/write MIDI files in a very friendly way
* `pyfluidsynth` + `fluidsynth` – turn MIDI into audio so we can **listen** to the result inside Colab

(If the audio tools fail to install, don't worry – the notebook has a fallback player.)

In [ ]:
!pip install -q pretty_midi pyfluidsynth
!apt-get install -y -qq fluidsynth > /dev/null 2>&1
print("Installation finished.")

## 3. Import libraries

In [ ]:
import os, glob, zipfile, json, random, urllib.request, warnings
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
import pretty_midi
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import IPython.display as ipd

warnings.filterwarnings("ignore")   # pretty_midi prints harmless warnings for some files

# Reproducibility: same seed -> same results (mostly) every run
SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.keras.utils.set_random_seed(SEED)

print("TensorFlow version :", tf.__version__)
print("GPUs available     :", tf.config.list_physical_devices("GPU"))

> The notebook works on a **normal CPU runtime**. If a GPU is available TensorFlow will use it automatically – no code changes needed.

## 4. Dataset

### What is MIDI?
**MIDI** (Musical Instrument Digital Interface) is *not* recorded sound. It is a list of **instructions**, like sheet music for a computer: *"play note 60 at time 1.0 s, release it at 1.5 s, with strength 80"*.

### Why MIDI instead of MP3/WAV?
| | MP3 / WAV | MIDI |
|---|---|---|
| Content | Raw sound waves (44,100 numbers per second!) | A short list of notes |
| Size of data | Huge | Tiny |
| Model needed | Very complex (spectrograms, WaveNet, …) | Simple sequence model (our LSTM) |
| Notes explicit? | No – hidden inside the waveform | Yes – pitch, start, end are given directly |

For a beginner project, MIDI lets us focus on **sequence modelling** instead of audio signal processing.

### What does a MIDI note contain?
| Field | Meaning | Example |
|---|---|---|
| `pitch` | Which key (0–127). 60 = middle C | 60 → C4, 62 → D4, 64 → E4 |
| `start` | When the note begins (seconds) | 1.0 |
| `end` | When the note ends (seconds) | 1.5 |
| `velocity` | How hard the key is struck (loudness, 0–127) | 80 |

### Our dataset: the Nottingham Folk Music Database
About 1,000 folk tunes, open-source, small (~1.6 MB zip), hosted on GitHub. Each file has a **melody track** and an **accompaniment (chords) track**. To keep the problem simple, we will model **only the melody track**.

### From MIDI to a sequence of musical events
For every note in the melody we create one **event** made of two things:

```
event = (pitch, duration)          e.g.  (64, 0.5)  →  "E4 lasting half a second"
```

Duration is **rounded to a few standard values** (sixteenth, eighth, quarter note …) so that the vocabulary stays small. A melody then becomes:

```
[(64, 0.5), (62, 0.5), (60, 1.0), (62, 0.5), ...]
```

> **Why include duration?** If we only predicted pitch, every note would have the same length and the music would sound like a robotic ringtone. With `(pitch, duration)` the model also learns basic rhythm.
> **Limitation (be aware!):** rests (silences) and chords are ignored in this simplified representation.

### Download the dataset

In [ ]:
DATA_URL = "https://github.com/jukedeck/nottingham-dataset/archive/refs/heads/master.zip"
DATA_DIR = "midi_data"
ZIP_PATH = "nottingham.zip"

if len(glob.glob(f"{DATA_DIR}/**/*.mid", recursive=True)) == 0:
    try:
        print("Downloading dataset ...")
        urllib.request.urlretrieve(DATA_URL, ZIP_PATH)
        with zipfile.ZipFile(ZIP_PATH) as z:
            z.extractall(DATA_DIR)
        print("Download + extraction complete.")
    except Exception as e:
        print("❌ Download failed:", e)
        print("➡ Use the manual upload cell below instead.")
else:
    print("Dataset already present – skipping download.")

midi_paths = sorted(glob.glob(f"{DATA_DIR}/**/*.mid", recursive=True) +
                    glob.glob(f"{DATA_DIR}/**/*.midi", recursive=True))
print("MIDI files found:", len(midi_paths))

### (Optional) Use your own MIDI files instead
If the download fails, or you want to use another style of music, run the cell below. It lets you upload a **ZIP file of `.mid` files**. Remove the leading `#` to activate it.

> If you use your own files, the code assumes the track with the **highest average pitch** is the melody. Single-instrument melody files work best.

In [ ]:
# from google.colab import files
# uploaded = files.upload()                       # choose your .zip file
# for name in uploaded:
#     with zipfile.ZipFile(name) as z:
#         z.extractall(DATA_DIR)
# midi_paths = sorted(glob.glob(f"{DATA_DIR}/**/*.mid", recursive=True) +
#                     glob.glob(f"{DATA_DIR}/**/*.midi", recursive=True))
# print("MIDI files found:", len(midi_paths))

## 5. Inspect the MIDI files

We start with a **subset** of the files (`MAX_FILES`) so that everything runs quickly. Increase it later for better results.

First, let's open *one* file and look at what is inside.

In [ ]:
MAX_FILES = 300          # use a subset; set to None for ALL files (slower, usually better)

example = pretty_midi.PrettyMIDI(midi_paths[0])
print("File:", midi_paths[0])
print("Number of instrument tracks:", len(example.instruments))
for i, inst in enumerate(example.instruments):
    print(f"  Track {i}: {len(inst.notes)} notes, pitch range "
          f"{min(n.pitch for n in inst.notes)}-{max(n.pitch for n in inst.notes)}")

print("\nFirst 10 notes of the melody track (track 0):")
print(f"{'pitch':>5} {'name':>5} {'start':>7} {'end':>7} {'duration':>9} {'velocity':>9}")
for n in sorted(example.instruments[0].notes, key=lambda n: n.start)[:10]:
    print(f"{n.pitch:>5} {pretty_midi.note_number_to_name(n.pitch):>5} {n.start:>7.2f} {n.end:>7.2f} "
          f"{n.end - n.start:>9.2f} {n.velocity:>9}")

Now let's load the melody track of every file in our subset. In this dataset the first track is usually the melody, but in some files the order is swapped, so we pick the track with the **highest average pitch** (melody notes sit above bass/chord notes). For each song we keep a list of `(pitch, start, end)` tuples, sorted by start time. Songs that are too short to make training sequences are skipped later.

In [ ]:
paths_to_use = midi_paths if MAX_FILES is None else midi_paths[:MAX_FILES]

songs_raw = []          # one entry per song: list of (pitch, start, end)
for path in paths_to_use:
    try:
        midi = pretty_midi.PrettyMIDI(path)
        if len(midi.instruments) == 0:
            continue
        # the melody is the track with the HIGHEST average pitch (the other track is bass/chords)
        melody_track = max([t for t in midi.instruments if len(t.notes) > 0],
                           key=lambda t: np.mean([n.pitch for n in t.notes]))
        melody_notes = sorted(melody_track.notes, key=lambda n: n.start)
        songs_raw.append([(n.pitch, n.start, n.end) for n in melody_notes])
    except Exception as e:
        print("Skipping", os.path.basename(path), "->", e)

song_lengths = [len(s) for s in songs_raw]
all_pitches  = [p for s in songs_raw for (p, _, _) in s]
all_durs     = [e - st for s in songs_raw for (_, st, e) in s]

print("Songs loaded          :", len(songs_raw))
print("Total notes           :", len(all_pitches))
print("Notes per song        : min =", min(song_lengths), " mean =", round(np.mean(song_lengths), 1),
      " max =", max(song_lengths))
print("Pitch range           :", min(all_pitches), "(", pretty_midi.note_number_to_name(min(all_pitches)), ") to",
      max(all_pitches), "(", pretty_midi.note_number_to_name(max(all_pitches)), ")")
print("Distinct pitches      :", len(set(all_pitches)))
print("Mean note duration (s):", round(np.mean(all_durs), 3))

fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
ax[0].hist(all_pitches, bins=range(min(all_pitches), max(all_pitches) + 2), color="steelblue")
ax[0].set_title("Pitch distribution (MIDI numbers)"); ax[0].set_xlabel("pitch"); ax[0].set_ylabel("count")
ax[1].hist(all_durs, bins=40, range=(0, 3), color="darkorange")
ax[1].set_title("Note durations (seconds)"); ax[1].set_xlabel("duration (s)")
plt.tight_layout(); plt.show()

## 6. MIDI preprocessing

### Step 6.1 – Turn each note into an *event* `(pitch, duration)`
We round every duration to the **nearest standard value**. The files are at 120 BPM, i.e. one beat (quarter note) = 0.5 s:

| Duration (s) | 0.25 | 0.5 | 0.75 | 1.0 | 1.5 | 2.0 |
|---|---|---|---|---|---|---|
| Musical meaning | eighth | quarter | dotted quarter | half | dotted half | whole |

The small helper function below does just this rounding.

In [ ]:
ALLOWED_DURATIONS = [0.25, 0.5, 0.75, 1.0, 1.5, 2.0]

def round_duration(d):
    # return the allowed duration that is closest to d
    return min(ALLOWED_DURATIONS, key=lambda x: abs(x - d))

# songs_events[i] is the list of (pitch, rounded_duration) events of song i
songs_events = []
for song in songs_raw:
    songs_events.append([(pitch, round_duration(end - start)) for (pitch, start, end) in song])

print("First 8 events of song 0:", songs_events[0][:8])

### Step 6.2 – Map every unique event to an integer ID
Neural networks work with numbers, not tuples. We build a **vocabulary**: every distinct event gets an integer.

```
(60, 0.5) → C4, half a beat   →  ID 17
(62, 0.5) → D4, half a beat   →  ID 23
...
```
We keep two lookup tables:
* `event_to_id` : event → integer (used when **preparing data**)
* `id_to_event` : integer → event (used when **generating music** and converting back to MIDI)

In [ ]:
all_events   = [ev for song in songs_events for ev in song]
unique_events = sorted(set(all_events))                 # sorted -> reproducible IDs

event_to_id = {ev: i for i, ev in enumerate(unique_events)}
id_to_event = {i: ev for ev, i in event_to_id.items()}
VOCAB_SIZE  = len(unique_events)

print("Total events      :", len(all_events))
print("Vocabulary size   :", VOCAB_SIZE, "(number of distinct (pitch, duration) events)")
print("\nSome mappings (event -> ID):")
for ev in unique_events[:6]:
    name = pretty_midi.note_number_to_name(ev[0])
    print(f"  pitch {ev[0]} ({name}), duration {ev[1]}s  ->  ID {event_to_id[ev]}")

# Convert every song to a list of integer IDs
songs_ids = [[event_to_id[ev] for ev in song] for song in songs_events]
print("\nSong 0 as integers:", songs_ids[0][:12], "...")

### Step 6.3 – Create `input sequence → next event` training examples
We slide a window of length `SEQ_LENGTH` over every song. The window is the **input**; the event right after the window is the **target**.

```
song:    [60, 62, 64, 67, 69, 67, 64, ...]      (shown as pitches for clarity)
                  SEQ_LENGTH = 4
example 1: input [60, 62, 64, 67]  →  target 69
example 2: input [62, 64, 67, 69]  →  target 67
example 3: input [64, 67, 69, 67]  →  target 64
```

> **Important detail:** we split into training and validation **by song** (not by random windows). Neighbouring windows overlap almost completely, so a random window split would let the model "peek" at validation data and give an over-optimistic validation loss.

In [ ]:
SEQ_LENGTH = 50          # how many past events the model sees (try 20, 50, 100)
VAL_FRACTION = 0.10      # 10% of songs are held out for validation

# keep only songs long enough to give at least one training example
usable_songs = [s for s in songs_ids if len(s) > SEQ_LENGTH]
print("Usable songs:", len(usable_songs), "of", len(songs_ids))

# split by song
rng = np.random.default_rng(SEED)
order = rng.permutation(len(usable_songs))
n_val_songs = max(1, int(VAL_FRACTION * len(usable_songs)))
val_songs   = [usable_songs[i] for i in order[:n_val_songs]]
train_songs = [usable_songs[i] for i in order[n_val_songs:]]

def make_sequences(list_of_songs, seq_length):
    X, y = [], []
    for song in list_of_songs:
        for i in range(len(song) - seq_length):
            X.append(song[i : i + seq_length])    # input window
            y.append(song[i + seq_length])        # the event that follows the window
    return np.array(X, dtype="int32"), np.array(y, dtype="int32")

X_train, y_train = make_sequences(train_songs, SEQ_LENGTH)
X_val,   y_val   = make_sequences(val_songs,   SEQ_LENGTH)

print("X_train shape:", X_train.shape, " (examples, SEQ_LENGTH)")
print("y_train shape:", y_train.shape, " (examples,)")
print("X_val   shape:", X_val.shape)

# Look at one example in human-readable form
print("\nExample input (first 6 events):",
      [pretty_midi.note_number_to_name(id_to_event[i][0]) for i in X_train[0][:6]], "...")
print("Its target event             :", pretty_midi.note_number_to_name(id_to_event[y_train[0]][0]),
      "for", id_to_event[y_train[0]][1], "s")

## 7. Prepare TensorFlow datasets

`tf.data.Dataset` feeds data to the model efficiently:

* **shuffle** – mix the training examples so batches are not in song order (helps training; *not* applied to validation)
* **batch** – process many examples at once (faster, smoother gradients)
* **prefetch** – prepare the next batch while the current one is training

**About input encoding:** we do *not* divide IDs by 127 or "normalize" them. IDs are *categories*, not quantities (ID 20 is not "twice as much" as ID 10). Instead, the **Embedding layer** in the model converts each integer ID into a learnable vector.

In [ ]:
BATCH_SIZE = 128

train_ds = (tf.data.Dataset.from_tensor_slices((X_train, y_train))
            .shuffle(buffer_size=10_000, seed=SEED)
            .batch(BATCH_SIZE)
            .prefetch(tf.data.AUTOTUNE))

val_ds = (tf.data.Dataset.from_tensor_slices((X_val, y_val))
          .batch(BATCH_SIZE)
          .prefetch(tf.data.AUTOTUNE))

for xb, yb in train_ds.take(1):
    print("One training batch:  inputs", xb.shape, " targets", yb.shape)

## 8. Build the LSTM

Architecture:

```
Input  (batch, SEQ_LENGTH)           ← integer IDs
  ↓
Embedding(VOCAB_SIZE, EMBEDDING_DIM) → (batch, SEQ_LENGTH, EMBEDDING_DIM)
  ↓
LSTM(LSTM_UNITS)                     → (batch, LSTM_UNITS)       ← only the LAST time-step output
  ↓
Dropout(0.2)                         → randomly switches off 20% of units while training (reduces overfitting)
  ↓
Dense(VOCAB_SIZE, softmax)           → (batch, VOCAB_SIZE)       ← probability of each possible next event
```

### Hyperparameters

| Parameter | Meaning |
|---|---|
| `SEQ_LENGTH` *(set in Section 6)* | How many past events the model can look at before predicting. Longer = more context, but slower and harder to train. |
| `EMBEDDING_DIM` | Size of the vector used to represent each event. The model learns these vectors; similar events end up with similar vectors. |
| `LSTM_UNITS` | Size of the LSTM's hidden/cell state – its "memory capacity". More units = more powerful but slower, and easier to overfit. |
| `BATCH_SIZE` *(set in Section 7)* | Number of examples processed before each weight update. |
| `EPOCHS` | Maximum number of passes through the whole training set (early stopping may stop sooner). |
| `LEARNING_RATE` | Step size of the optimiser. Too high → unstable; too low → very slow learning. |

In [ ]:
EMBEDDING_DIM = 64
LSTM_UNITS    = 256
EPOCHS        = 30
LEARNING_RATE = 0.002

model = keras.Sequential([
    layers.Input(shape=(SEQ_LENGTH,), dtype="int32"),
    layers.Embedding(input_dim=VOCAB_SIZE, output_dim=EMBEDDING_DIM),
    layers.LSTM(LSTM_UNITS),                               # returns only the final hidden state
    layers.Dropout(0.2),
    layers.Dense(VOCAB_SIZE, activation="softmax"),        # one probability per event in the vocabulary
], name="music_lstm")

model.summary()
print("Output units =", model.output_shape[-1], "| VOCAB_SIZE =", VOCAB_SIZE,
      "->", "match ✅" if model.output_shape[-1] == VOCAB_SIZE else "MISMATCH ❌")

## 9. Train the model

* **Loss:** `sparse_categorical_crossentropy`. Our target is a single integer (the correct next event), and the model outputs a probability distribution. The loss is `-log(probability the model gave to the correct event)`: it is small when the model is confident in the right answer. ("Sparse" means the targets are integers, not one-hot vectors.)
* **Optimiser:** Adam.
* **Early stopping:** stop when validation loss has not improved for 3 epochs, and restore the best weights. This prevents overfitting (memorising the training songs).

> Accuracy will look low (maybe 30–50 %). That is **normal** for music: many different next notes are musically acceptable, but we only reward the one that happened to be in the file.

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=3, restore_best_weights=True
)

history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=[early_stop])

### Loss curves
* **Training loss** going down → the model is learning the training data.
* **Validation loss** going down, then flattening or rising → after that point the model starts to *overfit*.

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(history.history["loss"],     label="training loss",   marker="o")
plt.plot(history.history["val_loss"], label="validation loss", marker="o")
plt.xlabel("epoch"); plt.ylabel("loss (sparse categorical crossentropy)")
plt.title("LSTM training curve"); plt.legend(); plt.grid(alpha=0.3)
plt.savefig("loss_curve.png", dpi=150, bbox_inches="tight")      # saved for your submission
plt.show()

print("Final training loss  :", round(history.history["loss"][-1], 4))
print("Final validation loss:", round(history.history["val_loss"][-1], 4))
print("Best validation loss :", round(min(history.history["val_loss"]), 4))
print("(A model guessing uniformly at random would have loss ln(VOCAB_SIZE) =", round(np.log(VOCAB_SIZE), 3), ")")

### Save the trained model (for your submission)
We also save the vocabulary, because the weights are useless without knowing which ID means which event.

In [ ]:
model.save("music_lstm.keras")
model.save_weights("music_lstm.weights.h5")
with open("vocabulary.json", "w") as f:
    json.dump({"seq_length": SEQ_LENGTH, "events": [list(ev) for ev in unique_events]}, f)
print("Saved: music_lstm.keras, music_lstm.weights.h5, vocabulary.json, loss_curve.png")

## 10. Next-event prediction: a probability distribution

The final **softmax** layer outputs one probability for *every* event in the vocabulary; they sum to 1. Schematically:

```
C4  → 0.05
D4  → 0.10
E4  → 0.20
G4  → 0.40   ← most likely
A4  → 0.15
...
```

Let's look at a *real* prediction from our model on a validation example, and compare it with what actually came next in the song.

In [ ]:
example_input  = X_val[10]                         # shape (SEQ_LENGTH,)
example_target = y_val[10]

probs = model.predict(example_input[np.newaxis, :], verbose=0)[0]   # add batch dim -> (1, SEQ_LENGTH)
print("Probabilities sum to:", round(float(probs.sum()), 4), "| number of probabilities:", len(probs))

def event_label(event_id):
    pitch, dur = id_to_event[int(event_id)]
    return f"{pretty_midi.note_number_to_name(pitch)} ({dur}s)"

top = np.argsort(probs)[::-1][:8]
print("\nTop 8 predicted next events:")
for i in top:
    print(f"  {event_label(i):>12} : {probs[i]:.3f}")
print("\nWhat actually came next in the song:", event_label(example_target),
      f"(model gave it probability {probs[example_target]:.3f})")

plt.figure(figsize=(8, 3.5))
plt.bar([event_label(i) for i in top], probs[top], color="seagreen")
plt.xticks(rotation=45, ha="right"); plt.ylabel("probability"); plt.title("Model's top-8 next-event probabilities")
plt.tight_layout(); plt.show()

### Why sample instead of always taking the `argmax`?
If we always pick the single most probable event (**greedy decoding**), then the same input always gives the same output. The generated melody quickly falls into a **loop** (e.g. the same few notes repeating forever), because the "safest" note keeps leading to the same "safest" note.

If instead we **sample** according to the probabilities (G4 chosen 40 % of the time, E4 20 %, …), we get **variety**, while still favouring musically likely notes. Let's see the difference on the same input:

In [ ]:
greedy_choice = int(np.argmax(probs))
sampled_choices = np.random.choice(len(probs), size=10, p=probs / probs.sum())

print("argmax  (always the same):", event_label(greedy_choice))
print("sampling (10 draws)      :", [event_label(i) for i in sampled_choices])

## 11. Temperature

Temperature reshapes the probability distribution **before** sampling. We take the logarithm of the probabilities, divide by the temperature `T`, and apply softmax again:

$$ p_i^{(T)} = \frac{\exp\big(\log(p_i)/T\big)}{\sum_j \exp\big(\log(p_j)/T\big)} $$

| Temperature | Effect | Musical result |
|---|---|---|
| `T < 1` (e.g. 0.5) | Distribution becomes **sharper** – likely notes become even more likely | Safe, predictable, but may be **repetitive** |
| `T = 1` | Original model probabilities | Balanced (default) |
| `T > 1` (e.g. 1.5) | Distribution becomes **flatter** | More diverse, surprising, but may sound **random / out of key** |

In [ ]:
def sample_with_temperature(probabilities, temperature=1.0):
    # 1. work in log-space and divide by temperature
    logits = np.log(np.asarray(probabilities, dtype="float64") + 1e-9) / temperature
    # 2. softmax to get a new valid probability distribution
    logits = logits - np.max(logits)                 # numerical stability
    new_probs = np.exp(logits) / np.sum(np.exp(logits))
    # 3. randomly pick one event ID according to the new probabilities
    return int(np.random.choice(len(new_probs), p=new_probs))

# Visualise how temperature changes the distribution for our example
plt.figure(figsize=(9, 3.5))
for T in [0.5, 1.0, 1.5]:
    adjusted = np.exp(np.log(probs + 1e-9) / T); adjusted /= adjusted.sum()
    plt.plot(np.sort(adjusted)[::-1][:15], marker="o", label=f"T = {T}")
plt.xlabel("events sorted by probability (rank)"); plt.ylabel("probability")
plt.title("Effect of temperature on the next-event distribution"); plt.legend(); plt.grid(alpha=0.3)
plt.show()

## 12. Generate music (autoregressive generation)

"Autoregressive" means the model's **own output becomes its next input**:

1. Start with a **seed** – a real sequence of `SEQ_LENGTH` events taken from the dataset.
2. Feed the sequence into the LSTM.
3. The LSTM outputs probabilities for the next event.
4. **Sample** the next event using the temperature.
5. **Append** it to the sequence.
6. **Remove the oldest** event so the window length stays `SEQ_LENGTH`.
7. Repeat for `num_generate` steps.

```
window: [e1 e2 e3 ... e50]  →  predict e51   →  window: [e2 e3 ... e50 e51]  →  predict e52 → ...
```

In [ ]:
def generate_events(model, seed_ids, num_generate=200, temperature=1.0):
    window = list(seed_ids)          # current input window (list of integer IDs)
    generated = []

    for _ in range(num_generate):
        x = np.array(window, dtype="int32")[np.newaxis, :]      # shape (1, SEQ_LENGTH)
        probabilities = model(x, training=False).numpy()[0]     # shape (VOCAB_SIZE,)
        next_id = sample_with_temperature(probabilities, temperature)

        generated.append(next_id)    # remember the new event
        window.append(next_id)       # add it to the end of the window ...
        window = window[1:]          # ... and drop the oldest event

    return generated

# ---- choose settings ----
NUM_GENERATE = 200       # how many NEW events (notes) to generate
TEMPERATURE  = 1.0       # 1.0 = default

# ---- choose a random seed from the validation songs ----
seed_song  = random.choice(val_songs)
start      = random.randint(0, len(seed_song) - SEQ_LENGTH)
seed_ids   = seed_song[start : start + SEQ_LENGTH]

generated_ids = generate_events(model, seed_ids, NUM_GENERATE, TEMPERATURE)

print("Generated", len(generated_ids), "events.")
print("First 10 generated events:", [event_label(i) for i in generated_ids[:10]])

## 13. Convert generated events back to MIDI

For every generated ID we:
1. look up the event `(pitch, duration)` with `id_to_event`,
2. create a `pretty_midi.Note` with
   * `start` = the time where the previous note ended (notes are placed one after another),
   * `end` = `start + duration`,
   * `velocity` = 80 (medium loudness),
3. add it to an instrument (program 0 = Acoustic Grand Piano),
4. write everything to `generated_music.mid`.

In [ ]:
def events_to_midi(event_ids, filename="generated_music.mid", velocity=80, program=0):
    pm = pretty_midi.PrettyMIDI(initial_tempo=120)
    instrument = pretty_midi.Instrument(program=program)

    current_time = 0.0
    for event_id in event_ids:
        pitch, duration = id_to_event[int(event_id)]
        note = pretty_midi.Note(velocity=velocity, pitch=int(pitch),
                                start=current_time, end=current_time + duration)
        instrument.notes.append(note)
        current_time += duration          # next note starts when this one ends

    pm.instruments.append(instrument)
    pm.write(filename)
    return pm

generated_midi = events_to_midi(generated_ids, "generated_music.mid")

print("File created:", os.path.exists("generated_music.mid"),
      "| size:", os.path.getsize("generated_music.mid"), "bytes")
print("Notes in file:", len(generated_midi.instruments[0].notes),
      "| length:", round(generated_midi.get_end_time(), 1), "seconds")

## 14. Play the result in Colab

Browsers cannot play `.mid` files directly, so we first **synthesize audio** from the MIDI notes.

1. **Preferred:** `fluidsynth` renders the MIDI with a real piano sound-font.
2. **Fallback:** if fluidsynth is unavailable, `pretty_midi.synthesize()` renders simple sine-wave tones (less pretty, but always works).

Then we display an audio player and a download button for the MIDI file.

In [ ]:
SAMPLE_RATE = 22050

def midi_to_audio(pm):
    try:
        return pm.fluidsynth(fs=SAMPLE_RATE)
    except Exception as e:
        print("fluidsynth not available (", type(e).__name__, ") -> using simple sine-wave synthesis instead.")
        return pm.synthesize(fs=SAMPLE_RATE)

audio = midi_to_audio(generated_midi)
ipd.display(ipd.Audio(audio, rate=SAMPLE_RATE))

In [ ]:
# Download button (works in Google Colab only)
try:
    from google.colab import files
    files.download("generated_music.mid")
except ImportError:
    print("Not running in Colab - the file is saved as generated_music.mid in the working folder.")

You can also download files manually: click the **📁 folder icon** in the left sidebar → right-click `generated_music.mid` → *Download*. To listen to the `.mid` on your own computer use VLC, MuseScore or an online MIDI player.

## 15. Experiments

Experiments are where you learn the most. Change **one thing at a time** and listen.

### 15.1 Temperature (no retraining needed)
The cell below generates melodies from the **same seed** with different temperatures and lets you listen to each one.

In [ ]:
temperature_files = {}
for T in [0.5, 1.0, 1.5]:
    ids = generate_events(model, seed_ids, NUM_GENERATE, T)
    fname = f"generated_T{T}.mid"
    pm = events_to_midi(ids, fname)
    temperature_files[T] = ids
    print(f"\n🎵 Temperature = {T}  ->  saved {fname}")
    ipd.display(ipd.Audio(midi_to_audio(pm), rate=SAMPLE_RATE))

### 15.2 Experiments that need retraining
Go back, change **one** value, and re-run from that section downwards.

| What to change | Where | What you should observe |
|---|---|---|
| `SEQ_LENGTH` (e.g. 10 → 50 → 100) | Section 6 | Short windows: music loses structure and wanders. Longer windows: more coherent phrases, slower training. Beyond a point, no further gain. |
| `LSTM_UNITS` (e.g. 32 → 128 → 512) | Section 8 | Small: underfits, high loss, bland melodies. Large: lower training loss, but validation loss may rise (overfitting) and the model may copy training tunes. |
| `TEMPERATURE` (0.3 → 1.0 → 2.0) | Section 12 | See 15.1: repetitive → natural → chaotic. |
| `NUM_GENERATE` (50 → 200 → 1000) | Section 12 | Long outputs show whether the melody stays coherent or drifts / gets stuck in loops. |
| `EPOCHS` (2 → 10 → 30; disable early stopping to see more) | Section 8 | Too few: random-sounding. Enough: sensible. Too many: validation loss rises (overfitting). |
| `MAX_FILES` (50 → 300 → all) | Section 5 | More data usually gives better generalisation. |

> Record your results (loss values + what you *heard*) in a table. It is required for the final assignment.

## 16. Evaluation

> ⚠️ **Loss alone does NOT tell us whether music is good.** A low loss means the model predicts the *next note of the training files* well. It says nothing about whether the generated piece sounds pleasant, original or coherent. A model that simply memorises the training songs can achieve a very low loss and still be a poor *composer*.

Music generation is evaluated using several complementary views:

| Aspect | Question |
|---|---|
| **Pitch distribution** | Does the generated music use similar notes (and similar range) as real music? |
| **Note diversity** | Does it use a healthy variety of notes, or only 3–4? |
| **Repetition** | Does it loop the same phrase endlessly (too repetitive), or never repeat anything (too random)? |
| **Rhythmic consistency** | Are note lengths sensible and consistent, or erratic? |
| **Musical coherence** | Does it have phrases, a key, a sense of direction? *(Needs human judgement.)* |
| **Human listening** | Does it sound musical and pleasant? **This is the primary evaluation for this assignment.** |

Below we compute **simple quantitative statistics** and compare generated music with *real* music from the dataset.

In [ ]:
def melody_stats(event_ids):
    # Simple numbers describing a melody given as a list of event IDs
    pitches = [id_to_event[int(i)][0] for i in event_ids]
    durs    = [id_to_event[int(i)][1] for i in event_ids]
    intervals = np.abs(np.diff(pitches))                             # jump between neighbouring notes
    fourgrams = [tuple(event_ids[i:i+4]) for i in range(len(event_ids) - 3)]
    repeated  = 1 - len(set(fourgrams)) / len(fourgrams)             # share of 4-note patterns seen before
    return {
        "unique pitches"          : len(set(pitches)),
        "pitch range (semitones)" : max(pitches) - min(pitches),
        "mean |interval|"         : round(float(np.mean(intervals)), 2),
        "unique durations"        : len(set(durs)),
        "mean duration (s)"       : round(float(np.mean(durs)), 3),
        "repeated 4-note patterns": round(float(repeated), 2),
    }

# "Real" reference: average over 30 random windows of real songs with the same length
real_stats_list = []
for _ in range(30):
    s = random.choice(usable_songs)
    if len(s) >= NUM_GENERATE:
        st = random.randint(0, len(s) - NUM_GENERATE)
        real_stats_list.append(melody_stats(s[st : st + NUM_GENERATE]))
real_avg = {k: round(float(np.mean([d[k] for d in real_stats_list])), 2) for k in real_stats_list[0]}

rows = {"REAL (dataset avg)": real_avg}
for T, ids in temperature_files.items():
    rows[f"generated, T={T}"] = melody_stats(ids)

keys = list(real_avg.keys())
print(f"{'':<22}" + "".join(f"{k:>26}" for k in keys))
for name, d in rows.items():
    print(f"{name:<22}" + "".join(f"{d[k]:>26}" for k in keys))

In [ ]:
# Pitch histograms: real data vs. generated music
fig, axes = plt.subplots(1, 3, figsize=(15, 3.5), sharey=True)
lo, hi = min(all_pitches), max(all_pitches)
axes[0].hist(all_pitches, bins=range(lo, hi + 2), density=True, color="steelblue")
axes[0].set_title("Real data (all songs)")
for ax, T in zip(axes[1:], [0.5, 1.5]):
    p = [id_to_event[i][0] for i in temperature_files[T]]
    ax.hist(p, bins=range(lo, hi + 2), density=True, color="tomato")
    ax.set_title(f"Generated, T = {T}")
for ax in axes: ax.set_xlabel("MIDI pitch")
axes[0].set_ylabel("relative frequency")
plt.tight_layout(); plt.show()

### How to read these numbers
* **Repeated 4-note patterns** close to 1 → very repetitive (typical at low temperature). Close to 0 → little structure (typical at high temperature). Real folk music sits in between – compare with the REAL row.
* **Pitch histogram:** at `T = 0.5` the model sticks to the most common notes; at `T = 1.5` it uses unusual pitches more often. Does the generated histogram resemble the real one?
* **Unique pitches / range:** far *more* than real data → the music is probably wandering randomly; far *fewer* → boring.

**Then listen!** Write down for each temperature: Is there a recognisable tune? Does it stay in one key? Do phrases repeat in a pleasant way? **Your ears are the final judge.**

## 17. Troubleshooting

| Problem | Likely cause | Fix |
|---|---|---|
| **"No MIDI files found"** / `midi_paths` is empty | Download failed or wrong folder | Re-run the download cell; check `!ls midi_data`; use the manual-upload cell. Make sure the files end in `.mid`/`.midi`. |
| **Dataset download failure** | GitHub temporarily unreachable / no internet | Retry after a minute; or download the ZIP on your computer and upload it with `files.upload()`. |
| **Memory error (RAM crash)** | Too many files, very large `SEQ_LENGTH` or `BATCH_SIZE` | Lower `MAX_FILES`, `SEQ_LENGTH`, `BATCH_SIZE`; use *Runtime → Restart session* and run again. |
| **Training is very slow** | CPU runtime, big model, lots of data | *Runtime → Change runtime type → GPU (T4)*. Reduce `LSTM_UNITS`, `MAX_FILES`, `EPOCHS`. |
| **Generated music is repetitive** | Temperature too low, model too small, too little training | Increase temperature (1.0–1.3), more units/data, train a little longer. |
| **Generated music sounds random** | Temperature too high, under-trained model | Lower temperature (0.7–1.0), train for more epochs, check that the validation loss is well below `ln(VOCAB_SIZE)`. |
| **MIDI doesn't play** | Browsers can't play `.mid`; fluidsynth missing | Use the audio player in Section 14 (it falls back to sine waves). Or open the `.mid` in VLC / MuseScore / an online MIDI player. |
| **`fluidsynth` import / `libfluidsynth` error** | apt package not installed in this session | Re-run the install cell (Section 2), then *Runtime → Restart session*. |
| **CUDA / GPU issues** (errors, "no GPU", cuDNN warnings) | GPU not assigned, or runtime mismatch | The notebook works on CPU – just switch to *Runtime → Change runtime type → CPU* (or *Disconnect and delete runtime* and reconnect). GPU warnings can usually be ignored. |
| **`ModuleNotFoundError: pretty_midi`** | Install cell not run in this session | Re-run Section 2 (Colab resets after a runtime restart). |
| **`ValueError` about shapes** | Changed `SEQ_LENGTH` but didn't re-run earlier cells | Re-run from Section 6 downwards (*Runtime → Run after*). |
| **Loading `.keras` model fails** | Different Keras/TF version | Rebuild the model (Section 8) and load `music_lstm.weights.h5` with `model.load_weights(...)`. |

## 18. Final Assignment

### Task
> **Train an LSTM music-generation model and generate an original MIDI sequence.**

### What to submit
1. **Colab notebook** (`.ipynb`) – fully executed, with all outputs visible, and your own comments/answers added.
2. **Trained model or weights** – `music_lstm.keras` (or `music_lstm.weights.h5`) **plus** `vocabulary.json`. *(If too large to upload, say so and include the training hyperparameters instead.)*
3. **Generated MIDI file** – `generated_music.mid` (at least ~30 seconds long).
4. **Loss graph** – `loss_curve.png` (training *and* validation loss).
5. **Explanation of the architecture** – ~1 paragraph: what each layer (Embedding, LSTM, Dropout, Dense + softmax) does, with tensor shapes, and why you chose your hyperparameters.
6. **Experiments with at least two different temperatures** – e.g. 0.5 and 1.2. Submit the MIDI file for each.
7. **Short observations** – 150–300 words: what you heard, the statistics from Section 16, and how the temperature changed the music.

### Suggested marking scheme (instructor may adapt)
| Component | Marks |
|---|---|
| Working end-to-end notebook (preprocessing → training → generation → MIDI) | 30 |
| Architecture explanation & hyperparameter justification | 15 |
| Loss curves + correct interpretation (overfitting / underfitting) | 10 |
| Temperature experiments (≥ 2 values) with evidence | 15 |
| Evaluation: statistics + listening comments | 15 |
| One additional experiment from Section 15.2 (e.g. `SEQ_LENGTH` or `LSTM_UNITS`) | 10 |
| Clarity of code and comments | 5 |

### Optional extensions (bonus ideas)
* Train on *all* files; compare with the 300-file subset.
* Stack two LSTM layers (`return_sequences=True` on the first).
* Replace `LSTM` with `SimpleRNN` or `GRU` and compare loss and sound.
* Add **rests** or **chords** to the event representation.
* Use your own MIDI dataset (e.g. classical piano).

---

## Why is an LSTM suitable for music, and why can a vanilla RNN struggle?

**Music has long-range structure.** A melody has a key, a repeated motif, phrases that answer each other – the "right" next note can depend on something played many steps earlier.

**Vanilla RNN.** An RNN carries a hidden state `h_t = tanh(W·h_{t-1} + U·x_t)`. When we train with backpropagation through time, the gradient must pass through this multiplication *once per time-step*. Over 50 or 100 steps it is multiplied again and again by numbers that are typically smaller than 1 (→ **vanishing gradient**) or larger than 1 (→ **exploding gradient**). In practice the network learns mainly from the last few steps and **forgets** older information.

**LSTM.** An LSTM adds a separate **cell state** (a "conveyor belt" of memory) controlled by three **gates**:
* **Forget gate** – what to erase from memory,
* **Input gate** – what new information to store,
* **Output gate** – what part of the memory to expose.

Because the cell state is updated mostly by *addition* instead of repeated squashing multiplications, gradients can flow across many time-steps almost unchanged. The network *learns* when to remember a note or a rhythm pattern and when to forget it – which is why LSTMs can capture musical context much better than a vanilla RNN.

**Limits to remember:** even an LSTM has limited memory (our window is only 50 events), our representation ignores rests, chords and dynamics, and a basic LSTM has no concept of global structure (verse/chorus). More advanced models such as Transformers (e.g. Music Transformer) handle those long-range dependencies better – a topic for later in your course.

🎶 *Well done – you built a neural network that composes music!*